<div style="font-size: 2em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
UE L2 Projet de Statistique CMI OSIA
</div>
<a href="https://www.u-bordeaux.fr/">
    <img src="https://upload.wikimedia.org/wikipedia/commons/e/e9/Universität_Bordeaux_Logo.svg?utm_source=fr.wikipedia.org&utm_campaign=index&utm_content=original"
         style="max-width: 180px; display: inline"
         alt="Université de Bordeaux"/>
</a>
<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Année 2026-2027
</div>
<div style="color: blue; font-size: 1.5em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Défi Science des données - Prévision de disponibilité de vélos dans des stations en libre service
</div>

<a href="https://www.python.org/">
    <img src="https://www.python.org/static/community_logos/python-logo.png"
         style="max-width: 180px; display: inline"
         alt="Python"/>
</a>

Cette UE est organisée sous la forme d'un défi en science des données qui propose l'utilisation de librairies en `Python` pour le traitement de données et l'implémentation de modèles statistiques pour une tâche d'apprentissage automatique de la prévision du nombre de vélos disponibles dans des stations en libre service dans différentes villes.

Dans ce projet, vous allez apprendre à :

- réaliser des analyses de statistique descriptive d'un jeu de données réelles de grande taille,
- concevoir différents modèles statistiques de prévisions de données temporelles et spatiales,
- mettre en place les principes de base de l'apprentissage statistique,
- implémenter des algorithmes de prévision et comparer leurs performances.

Les attendus du projet, qui est à réaliser en binôme, sont :

- un compte-rendu en .pdf qui présente votre démarche statistique, vos résultats de statistique descriptive des données, ainsi que les meilleurs modèles que vous avez retenus et leur validation à l'aide des données disponibles,

- un notebook Jupyter qui donne un code sous forme commentée conduisant à votre meilleur modèle,

- une soutenance orale de présentation des résultats de votre projet.

<div style="font-size: 2em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Prise en main des données
</div>

Les données de ce projet ont été collectées dans la ville de Toulouse par Max Halford, data scientist formé à l'Université de Toulouse : https://maxhalford.github.io

Ces données sont relatives au  nombre de vélos disponibles dans des stations de vélos en libre service, et elles ont été collectées sur plusieurs années de 2023 à 2026. Il s'agit d'un exemple d'un jeu de données ouvertes collectées en ligne. Les données ont été stockées dans des fichiers Parquet qui est un format de stockage très utilisé en Python pour la science des données. Pour ce projet vous avez accès à 4 fichiers :

- ``stations_toulouse.parquet`` qui contient la liste des stations où des données ont pu être collectées avec l'information sur leur nom et leur localisation géographique

- ``history_toulouse_train.parquet`` qui contient les données qui constituent la base d'apprentissage du défi. Ce fichier contient, pour chaque station où des mesures ont pu être réalisées, le nombre de vélos disponibles toutes les 15mn sur une plage de temps qui peut varier d'une station à l'autre

- ``history_toulouse_template.parquet`` qui contient les stations et les dates pour lesquelles vous devez faire des prévisions. Pour chaque station dans la base d'apprentissage, il est demandé de prévoir le nombre de vélos toutes les 15mn sur les 2 prochaines heures qui suivent la dernière date (jour/heure) de collecte de données, soit 8 prévisions à effectuer pour chaque station. 

<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Importation des données
</div>

Le code ci-dessous permet d'importer ces différents fichiers, de visualiser leur contenu ainsi que leur organisation sous forme de DataFrame Pandas.

In [1]:
import pandas as pd

stations_toulouse = pd.read_parquet("stations_toulouse.parquet")
stations_toulouse

,station,longitude,latitude,station_id
0,00001 - POIDS DE L'HUILE,1.445475,43.604190,0
1,00002 - LAFAYETTE,1.445306,43.604885,1
2,00003 - ALSACE - LORRAINE - POMME,1.445454,43.603302,2
3,00003 - ALSACE-LORRAINE - POMME,1.445454,43.603302,3
4,00003 - POMME,1.445454,43.603302,4
...,...,...,...,...
858,TOLOSANE - PASTOURELLES,1.477417,43.543115,858
859,TOLOSANE - POUCIQUOT,1.469875,43.551101,859
860,TOURNEFEUILLE MAIRIE,1.346042,43.583665,860
861,VELASQUEZ - TAMARIS,1.390347,43.621004,861


**Données d'entrainement**

``history_toulouse_train`` dans le code ci-dessous est un DataFrame Pandas, avec :
- des lignes identifiées par un MultiIndex (nom de la station et date de mesure)
- des colonnes nommées ``bikes`` et ``stands``qui contiennent respectivement le nombre de vélos disponibles et le nombre de vélos manquants pour une station et une date (jour/heure) données

In [2]:
history_toulouse_train = pd.read_parquet("history_toulouse_train.parquet")
history_toulouse_train

bikes  stands
station                  commit_at                         
00001 - POIDS DE L'HUILE 2023-08-05 12:15:00     10       9
                         2023-08-05 12:30:00     10       9
                         2023-08-05 12:45:00     10       9
                         2023-08-05 13:00:00     11       8
                         2023-08-05 13:15:00     11       8
...                                             ...     ...
TEST BEE                 2026-02-12 15:00:00      0       3
                         2026-02-12 15:15:00      0       3
                         2026-02-12 15:30:00      0       3
                         2026-02-12 15:45:00      0       3
                         2026-02-12 16:00:00      0       3

[40012622 rows x 2 columns]

**Données de test**

``history_toulouse_test`` dans le code ci-dessous est un DataFrame Pandas, avec :
- des lignes identifiées par un MultiIndex (une station à la date de la dernière mesure faite dans les données d'entraînement)
- des colonnes nommées ``bikes_+15min``,...,``bikes_+120min`` qui doivent être complétées par des prévisions tous les 15 minutes pendant 2 heures

In [3]:
history_toulouse_test = pd.read_parquet("history_toulouse_test.parquet")
history_toulouse_test

,horizon,bikes_+15min,bikes_+30min,bikes_+45min,bikes_+60min,bikes_+75min,bikes_+90min,bikes_+105min,bikes_+120min
station,commit_at,,,,,,,,
00001 - POIDS DE L'HUILE,2026-07-31 21:45:00,6,6,6,6,6,6,6,6
00002 - LAFAYETTE,2026-07-31 23:45:00,10,10,10,10,10,10,10,10
00003 - ALSACE - LORRAINE - POMME,2025-11-10 11:45:00,4,4,4,4,4,4,4,4
00003 - ALSACE-LORRAINE - POMME,2026-07-31 21:45:00,6,6,6,6,6,6,6,6
00003 - POMME,2025-01-05 05:45:00,10,10,10,10,10,10,10,10
...,...,...,...,...,...,...,...,...,...
NARBONNE - CLOTASSES,2025-08-22 19:15:00,0,0,0,0,0,0,0,0
PASTEUR - LAVOISIER,2025-08-05 10:15:00,0,0,0,0,0,0,0,0
PLACE GEORGES BRASSENS,2025-08-05 11:00:00,0,0,0,0,0,0,0,0


<div style="font-size: 2em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Travail à réaliser
</div>

**Statistique descriptive des données**

- nombre de stations de vélos et affichage sur une carte de leurs positions
- date de début et fin des mesures par stations
- présence de données manquantes ?
- statistique de base par station : maximimum, minimum, moyenne, écart-type, quartiles
- calcul de la courbe moyenne journalière de remplissage d'une station (toutes les 15 minutes), et de l'ensemble des stations / superposition de l'écart-type
- tracé sur une carte de l'évolution temporelle de remplissage moyen de l'ensemble des stations sur une journée -> quelles dynamiques urbaines ?
- autres idées ?...

**Méthode de prévision basiques**

- prévision par la moyenne du nombre de vélos disponibles
- prévision par la moyenne horaire (toutes les 15 mn) du nombre de vélos disponibles
- autres idées ?...
- soumissions de vos prévisions et cacul de l'erreur RMSE sur la partie publique des données de test


**Prévision par apprentissage statistique**

A venir...

<div style="font-size: 2em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Manipulation et visualisation des données
</div>

On présente ci-dessous quelques manipulations élémentaires et procédures de visualisation des données d'entraînement

In [4]:
# pip install folium
import folium

# Centre de la carte = moyenne des coordonnées
center_lat = stations_toulouse["latitude"].mean()
center_lon = stations_toulouse["longitude"].mean()

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=12
)

# Ajouter les stations
for _, row in stations_toulouse.iterrows():
    folium.Marker(
        location=[row["latitude"], row["longitude"]],
        popup=f"Station ID : {row['station']}",
        tooltip=f"Station {row['station']}"
    ).add_to(m)

m

ModuleNotFoundError: No module named 'folium'

In [ ]:
# Nom d'une station que l'on souhaite étudier
station = "00001 - POIDS DE L'HUILE"

# Sélection de toutes les observations de cette station
donnees_station = history_toulouse_train.loc[station]

Ici, ``.loc[station]`` permet de sélectionner les lignes correspondant à une valeur du premier niveau du MultiIndex (``station``).
On obtient alors un DataFrame dont l'index est uniquement commit_at

In [ ]:
print(donnees_station)

In [ ]:
# Récupérer uniquement les vélos disponibles

# On sélectionne la colonne "bikes"
bikes_station = donnees_station["bikes"]

print(bikes_station)

In [ ]:
# Idem
bikes_station = history_toulouse_train.loc[station, "bikes"]
print(bikes_station)

In [ ]:
# Tracer le nombre de vélos au cours du temps

import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

plt.plot(
    bikes_station.index,   # les dates de mesure
    bikes_station.values   # le nombre de vélos
)

plt.xlabel("Date de mesure")
plt.ylabel("Nombre de vélos")
plt.title(f"Nombre de vélos disponibles - {station}")

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Moyenne du nombre de vélos pour une station
moyenne_bikes = bikes_station.mean()

print(f"Nombre moyen de vélos : {moyenne_bikes:.2f}")

In [ ]:
# Autre statistiques
print(bikes_station.min())     # minimum
print(bikes_station.max())     # maximum
print(bikes_station.median())  # médiane
print(bikes_station.sum())     # somme
print(bikes_station.count())   # nombre de mesures
ecart_type_bikes = bikes_station.std()
print(f"Ecart-type : {ecart_type_bikes:.2f}")

In [ ]:
# Récupérer une mesure à une date donnée

station = "00001 - POIDS DE L'HUILE"
date = "2025-01-01 08:00:00"

try:
    mesure = history_toulouse_train.loc[(station, date)]

    print("Mesure trouvée :")
    print(mesure)

except KeyError:
    print(f"Aucune mesure disponible pour la station '{station}' "
          f"à la date {date}.")

date = "2020-01-01 08:00:00"

try:
    mesure = history_toulouse_train.loc[(station, date)]

    print("Mesure trouvée :")
    print(mesure)

except KeyError:
    print(f"Aucune mesure disponible pour la station '{station}' "
          f"à la date {date}.")

In [ ]:
# Les 5 premières observations de la station
donnees_station.head(5) 

In [ ]:
# Les 5 dernières observations de la station
donnees_station.tail(5)

In [ ]:
# Utilisation de groupby

# Récupèration de la dernière date de mesure pour chaque station
# des données d'entraînement
last = (
    history_toulouse_train
    .groupby(level="station")
    .tail(1)
    .copy()
)

last

In [ ]:
# Comparaison avec les données de test
history_toulouse_test

In [ ]:
# Date de la première mesure par station
first = (
    history_toulouse_train
    .groupby(level="station")
    .head(1)
)
first

In [ ]:
premieres_dates = first.index.get_level_values("commit_at")
dernieres_dates = last.index.get_level_values("commit_at")

In [ ]:
premieres_dates

In [ ]:
dernieres_dates

In [ ]:
# Calcul du maximum de vélos disponibles par station
max_velos = (
    history_toulouse_train
    .groupby(level="station")
    .max()
    .copy()
)

max_velos

In [ ]:
# Pour chaque station, on récupère son nombre maximum de vélos
max_bikes = max_velos["bikes"]

print(max_bikes.head())

In [ ]:
max_bikes = max_velos["bikes"]

stations_toulouse["nombre_max_velos"] = (
    stations_toulouse["station"].map(max_bikes)
)

In [ ]:
stations_toulouse

**Ajout du nombre maximum de vélos à chaque station avec `map()`**

Nous disposons de deux structures de données.

`stations_toulouse` est un DataFrame dans lequel le nom de la station est une **colonne** :

```python
stations_toulouse["station"]
```

Par exemple :

```text
0    00001 - POIDS DE L'HUILE
1    00002 - LAFAYETTE
2    00003 - ALSACE - LORRAINE - POMME
...
```

De son côté, `max_velos` est un DataFrame obtenu en regroupant les données par station :

```python
max_velos = (
    history_toulouse_train
    .groupby(level="station")
    .max()
)
```

Dans `max_velos`, le nom de la station n'est **pas une colonne** : c'est l'**index** du DataFrame :

```text
                                   bikes    stands
station
00001 - POIDS DE L'HUILE              20       19
00002 - LAFAYETTE                     25       24
00003 - ALSACE - LORRAINE - POMME     20       18
00003 - ALSACE-LORRAINE - POMME       19       18
00003 - POMME                          18       28
...
```

Lorsque nous écrivons :

```python
max_bikes = max_velos["bikes"]
```

nous sélectionnons uniquement la colonne `bikes`.

**`max_bikes` est alors une Series Pandas**, et non un DataFrame. Elle conserve l'index de `max_velos`, c'est-à-dire le nom des stations :

```text
station
00001 - POIDS DE L'HUILE              20
00002 - LAFAYETTE                     25
00003 - ALSACE - LORRAINE - POMME     20
00003 - ALSACE-LORRAINE - POMME       19
00003 - POMME                          18
...
Name: bikes, dtype: uint8
```

On peut voir cette structure comme un **dictionnaire** :

```text
nom de la station  →  nombre maximum de vélos

POIDS DE L'HUILE   →  20
LAFAYETTE          →  25
ALSACE - ...       →  20
...
```

La méthode `.map()` permet alors de rechercher, pour chaque station de `stations_toulouse`, la valeur correspondante dans `max_bikes` :

```python
stations_toulouse["nombre_max_velos"] = (
    stations_toulouse["station"].map(max_bikes)
)
```

Par exemple, pour la première ligne :

```text
stations_toulouse["station"]
        ↓
"00001 - POIDS DE L'HUILE"
        ↓
recherche dans l'index de max_bikes
        ↓
20
```

La valeur `20` est alors placée dans la nouvelle colonne `nombre_max_velos`.

Ainsi, `.map()` fait une **correspondance par valeur**, et non une correspondance par numéro de ligne. Il n'est donc pas nécessaire que les deux DataFrames soient dans le même ordre.

Une autre façon de voir `.map()` est :

> **Pour chaque valeur de la Series de gauche, chercher cette valeur dans l'index de la Series de droite et récupérer la valeur associée.**

Dans notre exemple :

```python
stations_toulouse["station"].map(max_bikes)
```

signifie :

> « Pour chaque nom de station de `stations_toulouse`, recherche ce nom dans l'index de `max_bikes` et récupère le nombre maximum de vélos correspondant. »

Si une station de `stations_toulouse` n'existe pas dans `max_bikes`, aucune correspondance n'est trouvée et Pandas place `NaN` dans la nouvelle colonne.


In [ ]:
import folium
import branca.colormap as cm

# Carte centrée sur les stations
m = folium.Map(
    location=[
        stations_toulouse["latitude"].mean(),
        stations_toulouse["longitude"].mean()
    ],
    zoom_start=12
)

# Échelle de couleurs basée sur le nombre maximum de vélos
colormap = cm.LinearColormap(
    colors=["blue", "cyan", "yellow", "orange", "red"],
    vmin=stations_toulouse["nombre_max_velos"].min(),
    vmax=stations_toulouse["nombre_max_velos"].max(),
    caption="Nombre maximum de vélos"
)

# Ajouter les stations
for _, row in stations_toulouse.dropna(
    subset=["nombre_max_velos"]
).iterrows():

    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=6,
        color=colormap(row["nombre_max_velos"]),
        fill=True,
        fill_color=colormap(row["nombre_max_velos"]),
        fill_opacity=0.8,
        popup=folium.Popup(
            f"""
            <b>Station :</b> {row['station']}<br>
            <b>ID :</b> {row['station_id']}<br>
            <b>Nombre maximum de vélos :</b>
            {row['nombre_max_velos']}
            """,
            max_width=300
        ),
        tooltip=(
            f"{row['station']} : "
            f"{row['nombre_max_velos']} vélos maximum"
        )
    ).add_to(m)

# Ajouter la légende
colormap.add_to(m)

m